# Part B — Text-Based Product Matching

**Goal:** decide whether two listings are the same product using **only their titles**.

```
Product titles  →  Text representation  →  Similarity (cosine)  →  similarity ≥ threshold ?  →  match / no match
```

**What we do in this notebook**
1. Turn titles into numbers (a *representation*) in four different ways and compare them:
   * **Baseline** — TF-IDF on words
   * **Experiment 1** — TF-IDF on word pairs (bigrams)
   * **Experiment 2** — TF-IDF on *characters* (n-grams)
   * **Experiment 3** — multilingual *sentence embeddings* (a pretrained neural network)
   * **Experiment 4** — Hybrid: Experiment 2 + Experiment 3 together
2. Choose the matching **threshold** and study the precision / recall trade-off.
3. Look at real **errors** and explain them.

**External resources that influenced this notebook:** scikit-learn's `TfidfVectorizer`; the `sentence-transformers` library and its pretrained
model `paraphrase-multilingual-MiniLM-L12-v2` (Reimers & Gurevych, 2019, *Sentence-BERT*); the public Kaggle *Shopee Product Matching* competition
(its discussions describe the same idea: compare title vectors with cosine similarity and keep neighbours above a threshold).

In [ ]:
# Install the extra library if the notebook environment does not have it yet
import importlib, subprocess, sys
for pkg, mod in [("sentence-transformers", "sentence_transformers")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

## 1. Setup — imports, settings and finding the dataset

This cell loads the libraries we need and finds the Shopee dataset automatically
(on Kaggle, Google Colab or your own computer — see `SETUP_GUIDE.md` for how to add the dataset).

In [ ]:
import os, re, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 100

SEED = 42                      # a fixed "random seed" so you get the same results every time you run
random.seed(SEED)
np.random.seed(SEED)

# ---------------- Settings you may want to change ----------------
N_GROUPS = 3000                # how many product groups to use (3000 groups ≈ 9,000 listings).
                               # Smaller = faster (use ~500 if you have no GPU). None = whole dataset (slow, needs lots of RAM).
RESULTS_DIR = "results"        # plots and tables are saved here
os.makedirs(RESULTS_DIR, exist_ok=True)


def find_data_dir():
    """Look for the folder that contains train.csv (works on Kaggle, Colab and a local computer)."""
    if os.environ.get("SHOPEE_DIR"):                       # you can force a path with this variable
        return os.environ["SHOPEE_DIR"]
    candidates = [
        "/kaggle/input/shopee-product-matching",           # Kaggle notebook (competition added via 'Add Input')
        "/kaggle/input/competitions/shopee-product-matching",
        "/content/shopee",                                 # Google Colab (see SETUP_GUIDE.md)
        "./shopee-product-matching", "./data", "../data", "../shopee-product-matching",
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, "train.csv")):
            return c
    for root, dirs, files in os.walk("/kaggle/input"):     # last resort: search everything under /kaggle/input
        if "train.csv" in files:
            return root
    raise FileNotFoundError("train.csv not found. Follow the steps in SETUP_GUIDE.md to add the dataset.")


DATA_DIR = find_data_dir()
IMG_DIR = os.path.join(DATA_DIR, "train_images")
print("Using data from:", DATA_DIR)

## 2. How we measure success (read this once — it is used in every experiment)

**The task, in one sentence:** for every listing, return the list of listings that are the *same product*.

We compare against the truth (`label_group`: same value = same product). For ONE listing:

| Symbol | Meaning |
|---|---|
| **returned** | how many listings our system says are the same product (it always includes the listing itself) |
| **true** | how many listings really are the same product (also includes itself) |
| **TP** | how many of the *returned* listings are really the same product |

* **Precision** = TP / returned  → "of the matches I claimed, how many were right?"
* **Recall** = TP / true  → "of the real matches, how many did I find?"
* **F1** = 2·TP / (returned + true)  → one number that balances precision and recall

The final score is the **average F1 over all listings**. This is exactly the metric of the original Kaggle competition.

**How a similarity score becomes a decision:** two listings are called the *same product* if `similarity ≥ threshold`.

**Why a "tune" pool and a "test" pool?**
We cannot choose the threshold on the same data we report the final score on — that would be cheating (the score would look
better than it really is). So we split the *product groups* into two pools:
* **tune pool (60 %)** — we try thresholds here and pick the best one,
* **test pool (40 %)** — we only *report* the score here, using the threshold chosen on the tune pool.

We split by *group* (not by listing) so that listings of one product never end up in both pools.

In [ ]:
# Thresholds we will try: 0.10, 0.12, ..., 1.00
THRESHOLDS = np.round(np.arange(0.10, 1.0001, 0.02), 2)


def sample_and_split(df, n_groups=N_GROUPS, test_frac=0.4, seed=SEED):
    """Pick `n_groups` random product groups and split those GROUPS into a 'tune' pool and a 'test' pool."""
    rng = np.random.RandomState(seed)
    groups = np.sort(df["label_group"].unique())
    if n_groups is not None and n_groups < len(groups):
        groups = rng.choice(groups, size=n_groups, replace=False)
    else:
        groups = groups.copy()
    rng.shuffle(groups)
    test_groups = groups[: int(len(groups) * test_frac)]
    sub = df[df["label_group"].isin(groups)].reset_index(drop=True)
    sub["split"] = np.where(sub["label_group"].isin(test_groups), "test", "tune")
    return sub


def sweep_thresholds(sim, labels, thresholds=THRESHOLDS):
    """
    sim    : N x N matrix, sim[i, j] = how similar listing i and listing j are (higher = more similar)
    labels : the true product group of each of the N listings
    For every threshold, return the average precision / recall / F1 over all listings,
    plus how many wrong matches (false positives) and missed matches (false negatives) there were.
    """
    same = labels[:, None] == labels[None, :]          # same[i, j] = True if i and j are truly the same product
    n_true = same.sum(axis=1)                          # number of true matches per listing (including itself)
    rows = []
    for t in thresholds:
        pred = sim >= t                                # our predicted matches
        np.fill_diagonal(pred, True)                   # a listing always matches itself
        n_pred = pred.sum(axis=1)                      # how many we returned per listing
        tp = (pred & same).sum(axis=1)                 # how many of those were correct
        rows.append({
            "threshold": t,
            "precision": float((tp / n_pred).mean()),
            "recall": float((tp / n_true).mean()),
            "f1": float((2 * tp / (n_pred + n_true)).mean()),
            "wrong_matches": int((n_pred - tp).sum()),      # false positives, counted per (listing, candidate)
            "missed_matches": int((n_true - tp).sum()),     # false negatives, counted per (listing, candidate)
        })
    return pd.DataFrame(rows)


def run_experiment(name, representation, similarity, sim_fn, sub, log, extra=None):
    """
    1) compute similarities inside the TUNE pool and pick the threshold with the best F1
    2) compute similarities inside the TEST pool and report the score at that threshold
    `sim_fn(idx)` must return the N x N similarity matrix for the listings `idx`.
    """
    t0 = time.time()
    labels = sub["label_group"].values
    tune_idx = np.where(sub["split"] == "tune")[0]
    test_idx = np.where(sub["split"] == "test")[0]
    sweep_tune = sweep_thresholds(sim_fn(tune_idx), labels[tune_idx])
    sweep_test = sweep_thresholds(sim_fn(test_idx), labels[test_idx])
    best = sweep_tune.loc[sweep_tune["f1"].idxmax()]
    thr = best["threshold"]
    at_thr = sweep_test[sweep_test["threshold"] == thr].iloc[0]
    row = {"Experiment": name, "Representation": representation, "Similarity": similarity,
           "Threshold": thr, "Tune F1": round(best["f1"], 4),
           "Test precision": round(at_thr["precision"], 4), "Test recall": round(at_thr["recall"], 4),
           "Test F1": round(at_thr["f1"], 4), "Seconds": round(time.time() - t0, 1)}
    if extra:
        row.update(extra)
    log.append(row)
    print(f"{name}: threshold={thr:.2f} | tune F1={best['f1']:.4f} | TEST F1={at_thr['f1']:.4f} "
          f"(precision {at_thr['precision']:.3f}, recall {at_thr['recall']:.3f})")
    return {"sweep_tune": sweep_tune, "sweep_test": sweep_test, "threshold": thr, "sim_fn": sim_fn}


def plot_threshold_curves(res, title, save_as=None):
    """Left: precision / recall / F1 versus threshold.  Right: number of wrong and missed matches."""
    st, se, thr = res["sweep_tune"], res["sweep_test"], res["threshold"]
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(st["threshold"], st["precision"], label="precision (tune)", color="tab:blue")
    ax[0].plot(st["threshold"], st["recall"], label="recall (tune)", color="tab:orange")
    ax[0].plot(st["threshold"], st["f1"], label="F1 (tune)", color="tab:green", lw=2.5)
    ax[0].plot(se["threshold"], se["f1"], "--", label="F1 (test)", color="tab:red")
    ax[0].axvline(thr, color="black", ls=":", label=f"chosen threshold = {thr:.2f}")
    ax[0].set_xlabel("threshold"); ax[0].set_ylabel("score"); ax[0].set_title(title); ax[0].legend(loc="center left", fontsize=8)
    ax[1].plot(st["threshold"], st["wrong_matches"] + 1, label="wrong matches (false positives)", color="tab:red")
    ax[1].plot(st["threshold"], st["missed_matches"] + 1, label="missed matches (false negatives)", color="tab:purple")
    ax[1].axvline(thr, color="black", ls=":")
    ax[1].set_yscale("log"); ax[1].set_xlabel("threshold"); ax[1].set_ylabel("count (log scale)")
    ax[1].set_title("Errors vs threshold (tune pool)"); ax[1].legend()
    plt.tight_layout()
    if save_as:
        plt.savefig(os.path.join(RESULTS_DIR, save_as), bbox_inches="tight")
    plt.show()

## 3. Load the data and build the working subset

`train.csv` has ~34,000 listings. Comparing every listing with every other one needs a 34,000 × 34,000 table
(~4.7 GB), so we work with a random **subset of product groups** (`N_GROUPS`). All the real structure is kept
(each group is complete), only the number of "distractor" listings is smaller, so scores will be a little
*higher* than on the full dataset. We say this clearly in the README.

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
print("Full training file:", df.shape)

sub = sample_and_split(df, N_GROUPS)
labels_all = sub["label_group"].values
titles = sub["title"].fillna("").astype(str).to_numpy(dtype=object)
tune_idx = np.where(sub["split"] == "tune")[0]
test_idx = np.where(sub["split"] == "test")[0]
print(f"Working subset: {len(sub):,} listings from {sub['label_group'].nunique():,} groups")
print(f"  tune pool: {len(tune_idx):,} listings | test pool: {len(test_idx):,} listings")
sub.head()

## 4. Text representations — what are we actually comparing?

A computer cannot compare words directly, so we turn each title into a **vector** (a list of numbers) and compare vectors.
* **Bag of words / TF-IDF:** one number per vocabulary word. *TF* = how often the word is in the title, *IDF* = how rare the word is in all titles
  (rare words like "Galaxy" count more than common words like "murah"). Two titles with many shared (rare) words get similar vectors.
* **Character n-grams:** same idea but on pieces of 3–5 *letters* ("sam", "amsu", "msung"). Robust to typos, spacing and abbreviations
  ("earbuds" ≈ "ear buds" ≈ "earbud").
* **Sentence embeddings:** a neural network reads the whole title and outputs a dense vector (here 384 numbers) in which titles with similar
  *meaning* are close — even when they use different words or languages.

**Cosine similarity** compares two vectors by the *angle* between them: 1 = same direction (very similar), 0 = nothing in common.
Because vectors are length-normalised, cosine similarity is simply a dot product.

Tiny example you can verify by hand:

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

toy = ["Samsung Galaxy Buds Pro Original", "Samsung Wireless Earbuds Pro", "Nike Running Shoes Black"]
toy_vec = TfidfVectorizer().fit_transform(toy)
toy_sim = (toy_vec @ toy_vec.T).toarray().round(2)
display(pd.DataFrame(toy_sim, index=["A", "B", "C"], columns=["A", "B", "C"]))
print("A and B share 'samsung' and 'pro' → some similarity. C shares nothing with A or B → 0.")
print("But 'Buds' and 'Earbuds' are almost the same word, and TF-IDF on words does NOT see that. Character n-grams and embeddings can.")

In [ ]:
def sparse_cosine(X, idx, chunk=1000):
    """Cosine similarity between all listings in `idx` for a TF-IDF matrix X (rows are already length-normalised)."""
    Xi = X[idx]
    out = np.zeros((len(idx), len(idx)), dtype=np.float32)
    for s in range(0, len(idx), chunk):          # in chunks to keep memory small
        out[s:s + chunk] = (Xi[s:s + chunk] @ Xi.T).toarray()
    return out


results = []        # one row per experiment — becomes our results table
experiments = {}    # keeps thresholds and curves of every experiment for the analysis below

## 5. Baseline — TF-IDF on words

**Why:** the simplest sensible text method: fast, no training, easy to explain. Every later experiment must beat this to be worth its cost.

**How:** `TfidfVectorizer()` lower-cases the title, splits it into words and weights them by TF-IDF. We fit it on all titles in our subset
(this uses **no labels**, so there is no leakage). Similarity = cosine. Threshold = the one with the best F1 on the tune pool.

In [ ]:
tfidf_word = TfidfVectorizer()
X_word = tfidf_word.fit_transform(titles)
print("Vocabulary size (number of different words):", len(tfidf_word.vocabulary_))

experiments["Baseline"] = run_experiment(
    "Baseline", "TF-IDF words (1-gram)", "Cosine", lambda idx: sparse_cosine(X_word, idx), sub, results,
    extra={"Dim": X_word.shape[1]})

## 6. Experiment 1 — TF-IDF on word pairs (1-grams + 2-grams)

**Hypothesis:** adding word pairs ("galaxy buds", "pro original") gives some word-order / phrase information, so matches become more precise.

**Only one thing changes** compared with the baseline: `ngram_range=(1, 2)`.

In [ ]:
tfidf_bi = TfidfVectorizer(ngram_range=(1, 2))
X_bi = tfidf_bi.fit_transform(titles)
print("Vocabulary size:", len(tfidf_bi.vocabulary_))

experiments["Exp 1: word 1-2 grams"] = run_experiment(
    "Exp 1: word 1-2 grams", "TF-IDF words (1-2 grams)", "Cosine", lambda idx: sparse_cosine(X_bi, idx), sub, results,
    extra={"Dim": X_bi.shape[1]})

## 7. Experiment 2 — TF-IDF on characters (3–5 letter n-grams)

**Hypothesis:** the same product is often written with typos, spaces, plurals or abbreviations ("earbuds / ear buds", "xiaomi / xiomi").
Character n-grams share many pieces even when whole words differ, so recall should go up.

`analyzer="char_wb"` builds n-grams inside word boundaries; `min_df=2` ignores n-grams that appear in only one title.

In [ ]:
tfidf_char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2)
X_char = tfidf_char.fit_transform(titles)
print("Number of character n-grams:", X_char.shape[1])

experiments["Exp 2: char n-grams"] = run_experiment(
    "Exp 2: char n-grams", "TF-IDF chars (3-5 grams)", "Cosine", lambda idx: sparse_cosine(X_char, idx), sub, results,
    extra={"Dim": X_char.shape[1]})

## 8. Experiment 3 — multilingual sentence embeddings

**Hypothesis:** a pretrained language model understands *meaning* and many languages, so "baju wanita" (Indonesian) and "women's shirt" end up
close together even though they share no letters — something TF-IDF can never do.

**Model:** `paraphrase-multilingual-MiniLM-L12-v2` — small and fast, supports 50+ languages (including Indonesian), outputs **384 numbers** per title.
We do **not** train it; we only use it to turn each title into a vector, then compare with cosine similarity.
(The first run downloads the model; on Kaggle switch "Internet" ON in the notebook settings.)

In [ ]:
from sentence_transformers import SentenceTransformer

cache_file = os.path.join(RESULTS_DIR, f"title_emb_minilm_{len(sub)}.npy")
if os.path.exists(cache_file) and np.load(cache_file).shape[0] == len(sub):
    emb_text = np.load(cache_file)
    print("Loaded cached embeddings", emb_text.shape)
else:
    t0 = time.time()
    st_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    emb_text = st_model.encode(list(titles), batch_size=128, show_progress_bar=True, normalize_embeddings=True)
    emb_text = np.asarray(emb_text, dtype=np.float32)
    np.save(cache_file, emb_text)
    print(f"Embedded {len(titles):,} titles in {time.time() - t0:.0f}s -> {emb_text.shape}")

if "st_model" not in globals():
    st_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")   # needed later for the two-title demo

experiments["Exp 3: sentence embeddings"] = run_experiment(
    "Exp 3: sentence embeddings", "MiniLM multilingual embeddings", "Cosine", lambda idx: emb_text[idx] @ emb_text[idx].T, sub, results,
    extra={"Dim": emb_text.shape[1]})

## 9. Experiment 4 — Hybrid (character TF-IDF + sentence embeddings)

**Hypothesis:** TF-IDF-style methods are good at *exact tokens* (model numbers, sizes, brand spelling) and embeddings are good at *meaning*.
They make different mistakes, so averaging the two similarities should beat each one alone.

`similarity = 0.5 × char-TF-IDF cosine + 0.5 × embedding cosine`. The 50/50 weight is a first guess (we tune weights properly in the Finale).

In [ ]:
def sim_hybrid(idx):
    return 0.5 * sparse_cosine(X_char, idx) + 0.5 * (emb_text[idx] @ emb_text[idx].T)


experiments["Exp 4: hybrid"] = run_experiment(
    "Exp 4: hybrid", "char TF-IDF + MiniLM", "0.5·cos + 0.5·cos", sim_hybrid, sub, results,
    extra={"Dim": f"{X_char.shape[1]} + {emb_text.shape[1]}"})

## 10. Results table

*Tune F1* = score on the tune pool at the best threshold (this is how we pick the winner).
*Test …* = score on the untouched test pool with that same threshold (this is the honest number to report).

In [ ]:
results_df = pd.DataFrame(results)
display(results_df[["Experiment", "Representation", "Dim", "Similarity", "Threshold", "Tune F1", "Test precision", "Test recall", "Test F1", "Seconds"]])
results_df.to_csv(os.path.join(RESULTS_DIR, "part_b_results.csv"), index=False)

plt.figure(figsize=(9, 4))
plt.barh(results_df["Experiment"][::-1], results_df["Test F1"][::-1], color="tab:blue")
for k, v in enumerate(results_df["Test F1"][::-1]):
    plt.text(v, k, f" {v:.3f}", va="center")
plt.xlabel("Test F1 (average over listings)"); plt.title("Text-only matching: test F1 by method")
plt.xlim(0, 1); plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "B_results_bar.png"), bbox_inches="tight")
plt.show()

best_name = results_df.loc[results_df["Tune F1"].idxmax(), "Experiment"]
print("Best method chosen on the TUNE pool:", best_name)

**How to read it:** compare each row with the baseline *using the Test F1 column*. Ask yourself:
* Did the more complex method actually help? By how much? Is the gain worth the extra time (`Seconds` column) and complexity?
* Look at precision vs recall for each method — did a method improve one at the cost of the other?

(Write your reading of the table in the Observations section at the bottom.)

## 11. Threshold analysis

A threshold turns a similarity score into a yes/no decision. It controls a trade-off:

| Threshold | More **false positives** (wrong matches) | More **false negatives** (missed matches) | Precision | Recall |
|---|---|---|---|---|
| **low** (accept weak similarity) | many | few | low | high |
| **high** (accept only very similar) | few | many | high | low |

F1 balances both. Below: the curves for the baseline and for the best method.

In [ ]:
plot_threshold_curves(experiments["Baseline"], "Baseline: TF-IDF words", save_as="B_threshold_baseline.png")
plot_threshold_curves(experiments[best_name], f"Best method: {best_name}", save_as="B_threshold_best.png")

In [ ]:
# Precision-recall curves of all methods in one picture (tune pool)
plt.figure(figsize=(7, 5))
for name, res in experiments.items():
    sw = res["sweep_tune"]
    plt.plot(sw["recall"], sw["precision"], label=name)
plt.xlabel("recall"); plt.ylabel("precision"); plt.title("Precision-recall trade-off (each point = one threshold)")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(os.path.join(RESULTS_DIR, "B_precision_recall.png"), bbox_inches="tight")
plt.show()

In [ ]:
sw = experiments[best_name]["sweep_tune"]
ops = {"Max F1 (our choice)": sw.loc[sw["f1"].idxmax()]}
hi_p = sw[sw["precision"] >= 0.9]
if len(hi_p):
    ops["Safest: precision >= 0.9"] = hi_p.loc[hi_p["recall"].idxmax()]
hi_r = sw[sw["recall"] >= 0.9]
if len(hi_r):
    ops["Widest net: recall >= 0.9"] = hi_r.loc[hi_r["precision"].idxmax()]
display(pd.DataFrame(ops).T[["threshold", "precision", "recall", "f1", "wrong_matches", "missed_matches"]].round(3))

**How we selected the final threshold:** the threshold with the **highest F1 on the tune pool** (the first row above),
because the competition metric is F1. We then *applied that fixed threshold to the test pool* — the dotted line in the left plot should sit
near the top of the red dashed (test) curve; if it does, the threshold generalises.

**When you would pick a different threshold:** if a wrong match is expensive (e.g. merging two different products on a shop page)
choose a *high-precision* operating point (second row). If missing a match is expensive (e.g. detecting duplicate listings to remove),
choose a *high-recall* operating point (third row).

## 12. Error analysis

We use the best method, its chosen threshold and the **test pool**. We look at three kinds of pairs:
* **Correct match (true positive):** same product and we said "match".
* **Wrong match (false positive):** different products but we said "match".
* **Missed match (false negative):** same product but we said "no match".

In [ ]:
chosen_thr = experiments[best_name]["threshold"]
sim_test = experiments[best_name]["sim_fn"](test_idx)
lab_test = labels_all[test_idx]
tit_test = titles[test_idx]

n_t = len(lab_test)
same_t = lab_test[:, None] == lab_test[None, :]
upper_t = np.triu(np.ones((n_t, n_t), dtype=bool), k=1)       # every pair only once
pred_t = sim_test >= chosen_thr

pair_sets = {
    "TP": np.where(upper_t & same_t & pred_t),
    "FP": np.where(upper_t & ~same_t & pred_t),
    "FN": np.where(upper_t & same_t & ~pred_t),
}
print(f"Method: {best_name} | threshold {chosen_thr:.2f} | test pool pairs:")
for k_, (a_, b_) in pair_sets.items():
    print(f"   {k_}: {len(a_):,}")


def pair_table(kind, how, k=8, seed=0):
    """how = 'highest' similarity first, 'lowest' similarity first, or 'random'."""
    ii, jj = pair_sets[kind]
    s = sim_test[ii, jj]
    if len(s) == 0:
        return pd.DataFrame()
    if how == "highest":
        pick = np.argsort(-s)[:k]
    elif how == "lowest":
        pick = np.argsort(s)[:k]
    else:
        pick = np.random.RandomState(seed).choice(len(s), size=min(k, len(s)), replace=False)
    return pd.DataFrame({"Title A": tit_test[ii[pick]], "Title B": tit_test[jj[pick]], "similarity": s[pick].round(3)})


pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 220)

print("\n✅ CORRECT matches (random examples):")
display(pair_table("TP", "random"))
print("\n❌ WRONG matches — the most confident ones (different products, but very similar titles):")
display(pair_table("FP", "highest"))
print("\n❌ WRONG matches — random examples:")
display(pair_table("FP", "random", k=6))
print("\n❌ MISSED matches — the lowest-similarity ones (same product, completely different titles):")
display(pair_table("FN", "lowest"))
print("\n❌ MISSED matches — random examples:")
display(pair_table("FN", "random", k=6))

**How to analyse each table** (write your own conclusions in the Observations section):
* *Correct matches:* what do the titles have in common that the method picked up on?
* *Wrong matches:* do the two titles differ only in a size, colour, quantity or model number? If so the method cannot see that one token matters.
* *Missed matches:* are the titles in different languages, very short, full of promo words, or describing the product from a different angle?

In [ ]:
# Quick numbers behind the tables
def numbers_in(title):
    return set(re.findall(r"\d+", title.lower()))


def words_in(title):
    return set(re.findall(r"[a-z]+", title.lower()))


ii, jj = pair_sets["FP"]
top_fp = np.argsort(-sim_test[ii, jj])[:1000]
differ = np.mean([numbers_in(tit_test[ii[p]]) != numbers_in(tit_test[jj[p]]) for p in top_fp]) if len(top_fp) else float("nan")
print(f"Among the {len(top_fp)} most confident wrong matches, the two titles contain DIFFERENT numbers in {100 * differ:.0f} % of the cases "
      f"(sizes, quantities, model numbers → likely product variants).")

ii, jj = pair_sets["FN"]
sample_fn = np.random.RandomState(0).choice(len(ii), size=min(2000, len(ii)), replace=False) if len(ii) else []
no_word = np.mean([len(words_in(tit_test[ii[p]]) & words_in(tit_test[jj[p]])) == 0 for p in sample_fn]) if len(sample_fn) else float("nan")
print(f"Among the missed matches, the titles share NO word at all in {100 * no_word:.0f} % of the cases.")

## 13. Using the system on two titles

The task asks for a system that takes **two listings** and says whether they are the same product. Here it is:

In [ ]:
def make_tfidf_scorer(vec):
    def score(a, b):
        M = vec.transform([a, b])
        return float(M[0].multiply(M[1]).sum())
    return score


def embed_score(a, b):
    E = st_model.encode([a, b], normalize_embeddings=True)
    return float(np.dot(E[0], E[1]))


char_score = make_tfidf_scorer(tfidf_char)
pair_scorers = {
    "Baseline": make_tfidf_scorer(tfidf_word),
    "Exp 1: word 1-2 grams": make_tfidf_scorer(tfidf_bi),
    "Exp 2: char n-grams": char_score,
    "Exp 3: sentence embeddings": embed_score,
    "Exp 4: hybrid": lambda a, b: 0.5 * char_score(a, b) + 0.5 * embed_score(a, b),
}


def are_same_product(title_a, title_b, method=best_name):
    score = pair_scorers[method](title_a, title_b)
    thr = experiments[method]["threshold"]
    return {"method": method, "similarity": round(score, 3), "threshold": round(float(thr), 2), "same product?": bool(score >= thr)}


print(are_same_product("Samsung Galaxy Buds Pro Original", "Samsung Wireless Earbuds Pro"))
print(are_same_product("Samsung Galaxy Buds Pro Original", "Nike Running Shoes Black"))
# and a real pair from the test pool
ii, jj = pair_sets["FP"]
if len(ii):
    print("A real wrong match:", are_same_product(tit_test[ii[0]], tit_test[jj[0]]))

## 14. Answers to the guiding questions

* **How should two titles be represented mathematically?** As vectors (sparse TF-IDF counts, or dense embeddings) compared by cosine similarity.
* **What makes TF-IDF effective / ineffective?** Effective: very fast, no training, rewards rare discriminative tokens (brand, model number).
  Ineffective: it only sees *exact* tokens — synonyms, typos, spacing and other languages look like different words, and one extra promo word shifts the score.
* **Would character n-grams help?** They are robust to typos / plurals / spacing, so they usually raise recall — see Exp 2 vs Baseline.
* **Why might semantic embeddings beat keywords?** They map meaning (and several languages) to nearby vectors, so listings that use different words
  for the same thing can still match. They can *lose* on exact details: "128GB" and "64GB" are almost the same to an embedding.
* **Very similar titles but different products?** The system produces false positives — see the "wrong match" tables. A single global threshold cannot fix this.
* **Same product with completely different titles?** The system produces false negatives — see the "missed match" tables. This is where the **image** has to help (Part C and the Finale).

## 15. My observations and conclusions  ✍️ (write these yourself after running)

1. **Baseline result:** TF-IDF words gave test F1 = … (precision …, recall …) at threshold …
2. **Which experiment helped most and why?** …
3. **Which experiment did *not* help, and why do I think so?** …
4. **Precision vs recall:** …
5. **Threshold:** I chose … because … The test curve shows that the threshold … (generalises / does not generalise).
6. **Typical false positives:** … (copy 1–2 real examples from the tables)
7. **Typical false negatives:** … (copy 1–2 real examples)
8. **Can titles alone do the job?** … (my answer, backed by the numbers above)
9. **What I would try next:** … (e.g. add the image, fine-tune the text model, give extra weight to numbers)